In [ ]:
using Base.Threads
println( "Number of threads: ", nthreads() )

include( "../args.jl" )
include( "../model.jl" )
include( "../geom.jl" )
include( "../recur.jl" )

loadsteps = false
savesteps = !loadsteps
loadsteps, savesteps

In [ ]:
# Social parameters.
ρ = 0.1;  β = 1.0

# System parameters.
μ = 1.0
N = defInt( 100 )
L = √(N/μ)

# Activity transition variables.
η = 1/500;  γ = 0.1;  τ = 75.0

# Movement variables.
ξ = 0.1;  λ = 0.50;  ϕ = 0.25
s = (2.5/4.0)*60^(ϕ + 1)

# Sensing area parameter.
α = (1/4)π

# Generate parameter variable.
scale = Scale( r, ξ )
params = Params(; ρ=ρ, η=η, β=β, γ=γ, τ=τ, ξ=ξ, λ=λ, ϕ=ϕ, s=s, α=α )
nondim = Nondim( params; scale=scale )
println( "   dim. parameters: ", params )
println( "nondim. parameters: ", nondim )

# Adapatable time-step length.
δt = Δt;  iτ = round( defInt, τ/δt );
(δt, iτ)

In [ ]:
case = "../data/results/case-0/"
if true
    if !isdir( case )
        mkpath( case )
    end
    saveparams( case*"default-params.json", params )
    savescale( case*"default-scale.json", scale )
end

# ςata folder name.
folder = case*"N-$(N)/mu-$(round( μ, digits=6 ))_xi-$(round( ξ, digits=6 ))/"

In [ ]:
# Run simulation under each environment parameter.
T = 500/scale.T;  Tload = 500/scale.T;  M = 100
Nt = round( defInt, T/δt );  tlist = 1:Nt
nt = round( defInt, 1/(2*δt*scale.T) );  tsave = Set( 1:nt:Nt );

# Frequency of adjacency calculation.
δt̂ = round( defInt, 0.1/δt );

In [ ]:
# Initialize list and run optimization.
xdata = [Matrix{defFloat}( undef, length( tsave ) + 1, 3 ) for _ ∈ 1:M]
zlist = Vector{State}( undef, M )
@threads for m ∈ 1:M
    # If steps are already saved, use as initial state.
    file = loadsteps ? folder*"steps/state_T-$(Tload)_m-$(m).txt" : nothing

    # Initialize agent states.
    z = initialstate( N, L/scale.L; A=1, file=file )
    ẑ = copystate( z )

    # Initialize adjacency and saved state.
    A = proximity( N, L/scale.L, nondim.r, nondim.α, z.x, z.y, z.θ )
    xdata[m][1,:] = statecomposition( N, ẑ )

    # Run simulation.
    t̂ = 2
    for t ∈ tlist
        # Update the adjacency matrix.
        (t % δt̂) == 0 && (A = proximity( N, L/scale.L, nondim.r, nondim.α, z.x, z.y, z.θ ))

        # Step simulation.
        step!( N, L/scale.L, nondim, z, ẑ; A=A, δt=δt )

        # Save state if in appropriate subset.
        t ∈ tsave && (xdata[m][t̂,:] = statecomposition( N, ẑ ); t̂ += 1)

        # Swap contents.
        tmp = z;  z = ẑ;  ẑ = tmp
    end

    # Save last simulation state.
    zlist[m] = z
end

In [ ]:
# Compute determinism metric and related statistics.
Rlist = Vector{RecurrenceMap}( undef, M )
ςlist = Vector{defFloat}( undef, M )
@threads for m ∈ 1:M
    Rlist[m] = recurrence( xdata[m]; δx=1/100 )
    ςlist[m] = determinism( Rlist[m]; ℓ0=15 )
end

# Determinism statistics.
ςmean = mean( ςlist )
ςstnd =  std( ςlist )
(ςmean, ςstnd)

In [ ]:
# Plot distribution of determinism metrics.
plt = plot( size=(400,250), xformatter=:plain, margin=10pt, dpi=600, legend=false )

histogram!( plt, ςlist; bins=0:0.05:1, range=(0,1), normalize=:probability, color=:gray )

plot!( plt; xlims=(0,1) )
plot!( plt; xlabel="determinism, "*L"ς", ylabel="pmf" )

In [ ]:
# Plot the mean activity deries for each duration value.
plt = plot( size=(400,200), xformatter=:plain, dpi=600 )
plot!( plt; right_margin=10pt )

m̂ = 10
for m ∈ 1:M
    alist = xdata[m][:,1]
    alpha = m == m̂ ? 1 : 1/25
    plot!( plt, δt*(0:nt:Nt), alist; color=:black, alpha=alpha, lw=2,
        label= m == m̂ ? latexstring( "ς = $(round( ςlist[m], digits=6 ))" ) : "" )
end

plot!( plt; xlims=(0,T), xlabel="time, "*L"t" )
plot!( plt; ylims=(0,1), ylabel="proportion of\nants active, "*L"a" )

In [ ]:
if (loadsteps || savesteps) && !isdir( folder*"/steps/" )
    mkpath( folder*"/steps/" )
end

if loadsteps
    writedlm( folder*"activity_T-$(round( defInt, T )).txt", [xlist[:,1] for xlist ∈ xdata] )
    writedlm( folder*"inactivity_T-$(round( defInt, T )).txt", [xlist[:,2] for xlist ∈ xdata] )
    writedlm( folder*"refractory_T-$(round( defInt, T )).txt", [xlist[:,3] for xlist ∈ xdata] )
    writedlm( folder*"determinism_T-$(round( defInt, T )).txt", ςlist )
    saveparams( folder*"sim-params.json", params )
    savescale( folder*"sim-scale.json", scale )
end

if savesteps
    for m ∈ 1:M
        savestate( folder*"steps/state_T-$(T)_m-$(m).txt", zlist[m] )
    end
end